In [0]:
import urllib.request, os, shutil

catalog = "workspace"
schema = "tp_transfermarkt"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.raw")

base = "https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/"
tablas = ["competitions", "clubs", "players", "games",
          "appearances", "player_valuations", "transfers"]
destino = f"/Volumes/{catalog}/{schema}/raw/transfermarkt/"
os.makedirs(destino, exist_ok=True)

for t in tablas:
    archivo = destino + f"{t}.csv.gz"
    req = urllib.request.Request(base + f"{t}.csv.gz",
                                 headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open(archivo, "wb") as f:
        shutil.copyfileobj(r, f)
    print(f"{t}: {os.path.getsize(archivo) / 1e6:.1f} MB")

competitions: 0.0 MB
clubs: 0.1 MB
players: 4.4 MB
games: 5.0 MB
appearances: 45.3 MB
player_valuations: 7.2 MB
transfers: 5.8 MB


In [0]:
destino = "/Volumes/workspace/tp_transfermarkt/raw/transfermarkt/"
tablas = ["competitions", "clubs", "players", "games",
          "appearances", "player_valuations", "transfers"]

for t in tablas:
    df = spark.read.option("header", True).csv(destino + f"{t}.csv.gz")
    print(f"\n=== {t}: {df.count():,} filas ===")
    print(", ".join(f"{c}" for c in df.columns))


=== competitions: 65 filas ===
competition_id, competition_code, name, sub_type, type, country_id, country_name, domestic_league_code, confederation, total_clubs, url

=== clubs: 796 filas ===
club_id, club_code, name, domestic_competition_id, total_market_value, squad_size, average_age, foreigners_number, foreigners_percentage, national_team_players, stadium_name, stadium_seats, net_transfer_record, coach_name, last_season, filename, url

=== players: 50,149 filas ===
player_id, first_name, last_name, name, last_season, current_club_id, player_code, country_of_birth, city_of_birth, country_of_citizenship, date_of_birth, sub_position, position, foot, height_in_cm, contract_expiration_date, agent_name, image_url, international_caps, international_goals, current_national_team_id, url, current_club_domestic_competition_id, current_club_name, market_value_in_eur, highest_market_value_in_eur

=== games: 88,958 filas ===
game_id, competition_id, season, round, date, home_club_id, away_club_

In [0]:
%sql
SELECT *
FROM workspace.tp_transfermarkt.bronze_player_valuations
WHERE player_id IS NULL
   OR try_cast(date AS TIMESTAMP) IS NULL
   OR try_cast(market_value_in_eur AS DOUBLE) IS NULL
   OR try_cast(market_value_in_eur AS DOUBLE) <= 0

player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id,_source_file,_ingested_at
60096,2008-03-17,0,NK Maribor,2696,RU1,dbfs:/Volumes/workspace/tp_transfermarkt/raw/transfermarkt/player_valuations.csv.gz,2026-10-22T00:21:48.287Z


In [0]:
%sql
SELECT 'silver_players: altura_razonable' AS regla, count(*) AS filas_que_fallan
FROM workspace.tp_transfermarkt.silver_players
WHERE height_in_cm IS NOT NULL AND height_in_cm NOT BETWEEN 140 AND 220
UNION ALL
SELECT 'silver_players: valor_no_negativo', count(*)
FROM workspace.tp_transfermarkt.silver_players
WHERE market_value_in_eur < 0
UNION ALL
SELECT 'silver_transfers: monto_no_negativo', count(*)
FROM workspace.tp_transfermarkt.silver_transfers
WHERE transfer_fee < 0
UNION ALL
SELECT 'silver_games: equipos_distintos', count(*)
FROM workspace.tp_transfermarkt.silver_games
WHERE home_club_id = away_club_id
UNION ALL
SELECT 'silver_clubs: estadio_con_capacidad_valida', count(*)
FROM workspace.tp_transfermarkt.silver_clubs
WHERE stadium_seats <= 0

regla,filas_que_fallan
silver_players: altura_razonable,13
silver_players: valor_no_negativo,0
silver_transfers: monto_no_negativo,0
silver_games: equipos_distintos,0
silver_clubs: estadio_con_capacidad_valida,1


In [0]:
%sql
SELECT 'jugador' AS tipo, name AS nombre, CAST(height_in_cm AS STRING) AS valor
FROM workspace.tp_transfermarkt.silver_players
WHERE height_in_cm NOT BETWEEN 140 AND 220
UNION ALL
SELECT 'club', name, CAST(stadium_seats AS STRING)
FROM workspace.tp_transfermarkt.silver_clubs
WHERE stadium_seats <= 0

tipo,nombre,valor
jugador,Fahad Al-Abdulrazzaq,17
jugador,Italo Vargas,18
jugador,Krzysztof Kurowski,18
jugador,Yesid Díaz,19
jugador,Genar Fornés,18
jugador,Connor Allan,19
jugador,Kilian Toscano,18
jugador,Adi Yona,17
jugador,Oliver Randazzo,18
jugador,Luis Marimón,18
